# Building a Neural Network


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

## Defining the neural network

In [3]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, input):
        c1 = F.relu(self.conv1(input))
        s2 = F.max_pool2d(c1, (2, 2))
        c3 = F.relu(self.conv2(s2))
        s4 = F.max_pool2d(c3, 2)
        s4 = torch.flatten(s4, 1)
        f5 = F.relu(self.fc1(s4))
        f6 = F.relu(self.fc2(f5))

        output = self.fc3(f6)
        return output

net = Net().to(device)
print(f"Using device: {device}")
print(net)

Using device: mps
Net(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (fc1): Linear(in_features=400, out_features=120, bias=True)
  (fc2): Linear(in_features=120, out_features=84, bias=True)
  (fc3): Linear(in_features=84, out_features=10, bias=True)
)


- Just have to define the forward pass function, the bacprop is handled by "autograd()" function
- Get the learnable parameters of a NN using "net.parameters()"

In [4]:
params = list(net.parameters())
print(len(params))
print(params[0].size())     ##conv1's weights

10
torch.Size([6, 1, 5, 5])


In [5]:
##move input to(device), input and model should be on the same device
input = torch.randn(1, 1, 32, 32).to(device)    ##torch.randn(barch_size, channels, height, width)
out = net(input)
print(out)

tensor([[-0.1498, -0.0004,  0.0667, -0.0849, -0.0339, -0.0472,  0.0745,  0.0182,
         -0.0871, -0.1356]], device='mps:0', grad_fn=<LinearBackward0>)


In [6]:
net.zero_grad()
out.backward(torch.randn(1, 10).to(device))

## Loss Function

In [7]:
output = net(input)
target = torch.randn(10).to(device)        ##dummy target
target = target.view(1, -1)     ##make target of same shape as output, 1 means 1 row and -1 specifies torch to calculate the column value itself, in this exmpale its 10
criterion = nn.MSELoss()

loss = criterion(output, target)
print(loss)

tensor(2.3624, device='mps:0', grad_fn=<MseLossBackward0>)


In [8]:
print(loss.grad_fn)
print(loss.grad_fn.next_functions[0][0])    ##"next_functions" contains references to the operations that came before the current one
print(loss.grad_fn.next_functions[0][0].next_functions[0][0])

## Backprop

In [9]:
net.zero_grad()

print("Conv1 bias grad before backward: ")
print(net.conv1.bias.grad)

loss.backward()

print("Conv1 bias grad after backward: ")
print(net.conv1.bias.grad)


Conv1 bias grad before backward: 
None
Conv1 bias grad after backward: 
tensor([-0.0080, -0.0229,  0.0081,  0.0254,  0.0029,  0.0142], device='mps:0')


## Updating the weights

In [10]:
##weight = weight - learning_rate * gradient (basic gradient descent)

learning_rate = 0.01
for f in net.parameters():
    with torch.no_grad():
        f -= f.grad * learning_rate